# ER-CyRIS Siklus 2  Audit terarah M3 dengan gangguan penjelasan yang terpisah

**Tujuan.** Menguji apakah skor stabilitas SHAP `r_x` dari v2 sejalan dengan kestabilan fitur penjelas teratas pada dua gangguan numerik baru yang tidak dipakai untuk menghitung `r_x`. Perbandingan gerbang margin saja versus margin + SHAP menggunakan **ambang v2 yang tetap**. Ini memeriksa konsistensi penjelasan pada model dan gangguan tertentu; **bukan** uji kebenaran alasan serangan, fidelity terhadap dunia nyata, kegunaan manusia, keberhasilan M4, atau kesiapan M5–M6.

Input: `UNSW-NB15_1.csv`, lock dan split yang sama, hasil v2, serta dua hasil M2 yang baru ditinjau. File2 tidak dibuka. Semua hasil development sebelumnya sudah pernah dilihat sehingga temuan di development bersifat **eksploratif**. Simpan keluaran kecil saja: JSON dan CSV gzip baru. Arsip v2 dan M2 tidak diubah.

**Prasyarat lokasi Colab Drive:** tempatkan `unsw_m2_stability_summary.json` dan `unsw_m2_stability_instances.csv.gz` pada `/content/drive/MyDrive/ercyris_siklus2/unsw_m2_stability_file1_v1/`. Hasil v2 tetap pada folder `unsw_framework_dev_v2/` sebagaimana notebook sebelumnya. Jika nama folder pengguna berbeda, hanya ubah variabel `M2DIR` di sel berikut.


In [ ]:
import csv, hashlib, json, platform
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, f1_score, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import shap
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/ercyris_siklus2')
except ImportError:
    BASE=Path('.')
P1=BASE/'UNSW-NB15_1.csv'
LOCK_DIR=BASE/'unsw_locked_protocol_v1'
LOCK=LOCK_DIR/'unsw_lock_summary.json'
MANIFEST=LOCK_DIR/'unsw_locked_split_manifest.csv'
V2FILE=BASE/'unsw_framework_dev_v2'/'unsw_framework_development_v2.json'
M2DIR=BASE/'unsw_m2_stability_file1_v1'
M2SUM=M2DIR/'unsw_m2_stability_summary.json'
M2CSV=M2DIR/'unsw_m2_stability_instances.csv.gz'
OUT=BASE/'unsw_m3_explanation_independent_file1_v1'
OUTFILE=OUT/'unsw_m3_independent_summary.json'
EVIDFILE=OUT/'unsw_m3_independent_instances.csv.gz'
SEED=42; MISSING_RATE=.10; PROBE_SIGMA=.01; SHAP_MAX=300; TOPK=10
TAU_Q=.95; TAU_Q_SEVERE=.70; MIN_GATE_COVERAGE=.50
for path in (P1,LOCK,MANIFEST,V2FILE,M2SUM,M2CSV):
    if not path.is_file(): raise FileNotFoundError(f'Dibutuhkan: {path}')
if OUTFILE.exists() or EVIDFILE.exists():raise FileExistsError('Keluaran M3 sudah ada; STOP agar tidak menimpa.')
print('Output:',OUTFILE,'| Hanya file1.')

## 1  Periksa identitas file1 dan partisi terkunci

Pemeriksaan menghentikan eksekusi jika data, jumlah baris, label, atau indeks sumber berbeda dari lock sebelumnya. Baris holdout tetap tidak dimuat.


In [ ]:
COLS=['srcip','sport','dstip','dsport','proto','state','dur','sbytes','dbytes',
      'sttl','dttl','sloss','dloss','service','sload','dload','spkts','dpkts',
      'swin','dwin','stcpb','dtcpb','smeansz','dmeansz','trans_depth',
      'res_bdy_len','sjit','djit','stime','ltime','sintpkt','dintpkt',
      'tcprtt','synack','ackdat','is_sm_ips_ports','ct_state_ttl',
      'ct_flw_http_mthd','is_ftp_login','ct_ftp_cmd','ct_srv_src','ct_srv_dst',
      'ct_dst_ltm','ct_src_ltm','ct_src_dport_ltm','ct_dst_sport_ltm',
      'ct_dst_src_ltm','attack_cat','label']
EXCLUDE={'label','attack_cat','srcip','dstip','stime','ltime'}
FEATURES=[c for c in COLS if c not in EXCLUDE]
CATS=['proto','state','service'];NUMS=[c for c in FEATURES if c not in CATS]
assert len(FEATURES)==43

def sha256(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for piece in iter(lambda:f.read(4*1024*1024),b''):h.update(piece)
    return h.hexdigest()

lock=json.loads(LOCK.read_text(encoding='utf-8'))
if lock['status']!='FROZEN_SPLIT_NO_MODEL_NO_HOLDOUT_EVALUATION':
    raise ValueError('Status lock tidak sesuai; STOP.')
if sha256(P1)!=lock['sources']['file1']['sha256'] or sha256(MANIFEST)!=lock['manifest']['sha256']:
    raise ValueError('Berkas sumber atau manifest berubah sejak lock; STOP.')
manifest=pd.read_csv(MANIFEST,dtype={'feature_group_hash64':str})
if len(manifest)!=lock['manifest']['row_count']:
    raise ValueError('Jumlah baris manifest berubah; STOP.')
local=manifest.loc[manifest.source_file==P1.name].copy()
if set(local.partition)!={'train','calibration','development'} or len(local)!=30000:
    raise ValueError('Partisi file1 salah; STOP.')
if manifest.loc[manifest.partition=='holdout','source_file'].nunique()!=1:
    raise ValueError('Holdout bukan tepat satu file; STOP.')
if local.source_row_zero_based.duplicated().any():
    raise ValueError('Nomor baris file1 duplikat lintas partisi; STOP.')
with P1.open('r',encoding='utf-8-sig',errors='replace',newline='') as f:
    r=next(csv.reader(f))
if len(r)!=49 or r[-1].strip() not in ('0','1'):
    raise ValueError('Skema sumber tidak cocok; STOP.')
needed=set(local.source_row_zero_based.astype(int)); chunks=[]; offset=0
for chunk in pd.read_csv(P1,header=None,names=COLS,dtype=str,na_filter=False,chunksize=100000):
    positions=np.arange(offset,offset+len(chunk),dtype=np.int32)
    yes=np.isin(positions,list(needed))
    if yes.any():
        z=chunk.loc[yes].copy();z.index=positions[yes];chunks.append(z)
    offset+=len(chunk)
if offset!=lock['sources']['file1']['n']:raise ValueError('Jumlah baris file1 berbeda.')
raw=pd.concat(chunks).sort_index()
if len(raw)!=len(local):raise ValueError('Baris dalam manifest tidak seluruhnya ditemukan.')
labels=pd.to_numeric(raw['label'],errors='coerce')
if labels.isna().any() or not labels.isin([0,1]).all():raise ValueError('Label file1 tidak biner.')
lookup=local.set_index('source_row_zero_based').sort_index()
if not np.array_equal(labels.to_numpy(dtype=int),lookup['label'].to_numpy(dtype=int)):
    raise ValueError('Label manifest tidak sama dengan label sumber.')
for k in ('train','calibration','development'):
    print(k,lock['partitions'][k]['n'],'positif',lock['partitions'][k]['positives'])
print('Manifest tervalidasi. File2 belum dibaca.')

## 2  Rekonstruksi fitur dan dua model tanpa mengubah konfigurasi

Transformasi hanya dipelajari pada train. Nilai hilang 10% memakai benih yang sama seperti hasil v2 dan M2.


In [ ]:
def prepare(frame):
    x=frame[FEATURES].copy()
    for c in NUMS:x[c]=pd.to_numeric(x[c],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for c in CATS:x[c]=x[c].astype(str).replace('', '__MISSING__')
    return x

def evidence_completeness(x):
    present=x[FEATURES].notna().copy()
    for c in CATS:present[c]&=x[c].astype(str).ne('__MISSING__')
    return present.mean(axis=1).to_numpy(dtype=float)

def synthetic_missing(frame, seed, frac=MISSING_RATE):
    v=frame.copy();rng=np.random.default_rng(seed)
    mask=rng.random((len(v),len(NUMS)))<frac
    for k,c in enumerate(NUMS):v.loc[mask[:,k],c]=np.nan
    return v

parts={name:lookup.loc[lookup.partition==name].index.to_numpy(dtype=np.int32)
       for name in ('train','calibration','development')}
xraw={name:prepare(raw.loc[ix]) for name,ix in parts.items()}
y={name:labels.loc[ix].to_numpy(dtype=np.int8) for name,ix in parts.items()}
xraw['cal_missing']=synthetic_missing(xraw['calibration'],SEED+1)
xraw['dev_missing']=synthetic_missing(xraw['development'],SEED+2)
y['cal_missing']=y['calibration'];y['dev_missing']=y['development']
q={name:evidence_completeness(xraw[name]) for name in xraw}
try:
    enc=OneHotEncoder(handle_unknown='ignore',sparse_output=False,dtype=np.float32)
except TypeError:
    enc=OneHotEncoder(handle_unknown='ignore',sparse=False,dtype=np.float32)
prep=ColumnTransformer([
    ('numeric',Pipeline([('median',SimpleImputer(strategy='median')),
                         ('scale',MinMaxScaler())]),NUMS),
    ('categorical',enc,CATS)
],remainder='drop',sparse_threshold=0)
X={'train':np.asarray(prep.fit_transform(xraw['train']),dtype=np.float32)}
for name in xraw:
    if name!='train':X[name]=np.asarray(prep.transform(xraw[name]),dtype=np.float32)
if any(not np.isfinite(v).all() for v in X.values()):raise ValueError('Vektor fitur nonhingga.')
if len({v.shape[1] for v in X.values()})!=1:raise ValueError('Dimensi antarpartisi berbeda.')
print('Fitur hasil train-only transform:',X['train'].shape[1])
print('q_e completeness clean calibration:',float(q['calibration'].mean()),
      'missingness synthetic:',float(q['cal_missing'].mean()))

## 3  Definisi uji M3

Indeks sepuluh fitur dengan nilai absolut SHAP terbesar dibandingkan antara alert asal dan masing-masing dari dua gangguan numerik independen. Kemiripan himpunan dihitung dengan Jaccard. Karena penjelasan untuk prediksi yang berubah tidak mencerminkan pertanyaan yang sama, hasil utama adalah rerata Jaccard **hanya pada alert yang tetap alert dalam kedua gangguan**. Jumlah alert yang gagal bertahan dilaporkan terpisah. Angka ini mengukur konsistensi fitur penjelas menurut protokol sintetis, bukan fidelitas atau relevansi sebab ancaman.

Gangguan dibuat pada fitur numerik hasil transformasi dengan σ=0,01 **tanpa clipping**. Skor M3 lama dalam CSV masih dihitung dengan probe v2 yang memakai clipping; perbedaan itu dilaporkan sebagai batas eksperimen. Benih baru tidak sama dengan v2 (`SEED+5`) ataupun rangkaian M2. Tidak ada ambang baru yang ditetapkan dari hasil uji ini.


In [ ]:
def select_detector_threshold(y_true,p):
    candidates=np.linspace(.05,.95,19)
    scored=[]
    for t in candidates:
        pred=(p>=t).astype(int)
        tn,fp,fn,tp=confusion_matrix(y_true,pred,labels=[0,1]).ravel()
        f=f1_score(y_true,pred,zero_division=0)
        far=float(fp/(fp+tn)) if fp+tn else 1.
        scored.append((float(f),-far,-abs(float(t)-.5),float(t)))
    return max(scored)[3]

def margin_d(probability,detector_threshold):
    p=np.clip(np.asarray(probability,dtype=float),0.,1.)
    return np.clip((p-detector_threshold)/max(1.-detector_threshold,1e-12),0.,1.)

def stable_seed(*labels):
    payload='|'.join(map(str,(SEED,'M3_INDEPENDENT_V1')+labels)).encode('utf-8')
    return int.from_bytes(hashlib.sha256(payload).digest()[:8],'little')

def jitter_numeric_without_clipping(xmat,seed,sigma=.01):
    out=np.asarray(xmat,dtype=np.float32).copy()
    noise=np.random.default_rng(seed).normal(0,sigma,(len(out),len(NUMS))).astype(np.float32)
    out[:,:len(NUMS)]+=noise
    return out

def tree_shap_positive(explainer,x):
    try:v=explainer.shap_values(x,check_additivity=False)
    except TypeError:v=explainer.shap_values(x)
    if isinstance(v,list):v=v[1]
    v=np.asarray(v,dtype=float)
    if v.ndim==3:
        if v.shape[-1]!=2:raise ValueError('Bentuk SHAP biner tidak sesuai; STOP.')
        v=v[:,:,1]
    if v.shape!=x.shape or not np.isfinite(v).all():raise ValueError('SHAP tidak valid; STOP.')
    return v

def topk_jaccard(base,probe,k=10):
    if base.shape!=probe.shape or base.ndim!=2:raise ValueError('Matriks SHAP tidak kompatibel.')
    k=min(k,base.shape[1]);a=np.argsort(-np.abs(base),axis=1)[:,:k]
    b=np.argsort(-np.abs(probe),axis=1)[:,:k]
    return np.array([len(set(x)&set(y))/len(set(x)|set(y)) for x,y in zip(a,b)],dtype=float)

def gate2(margin,shap_score,tau_d,tau_x):
    a=np.asarray(margin)>=tau_d;b=np.asarray(shap_score)>=tau_x
    return np.where(a&b,'accept',np.where(a|b,'warn','abstain'))


## 4  Pagar identitas hasil v2 dan M2

Ambang deteksi, kebingungan, F1, dan AP harus identik dengan v2 sebelum uji SHAP baru dapat berjalan. CSV per alert harus cocok dengan hash dalam ringkasan M2 dan dengan identitas prediksi yang baru direkonstruksi.


In [ ]:
models={
 'XGBoost':XGBClassifier(random_state=SEED,n_jobs=-1,eval_metric='logloss',verbosity=0,tree_method='hist'),
 'RandomForest':RandomForestClassifier(random_state=SEED,n_jobs=-1),
}
trained={};thresholds={};probs={};detector_scores={}
for name,model in models.items():
    model.fit(X['train'],y['train']);trained[name]=model
    pcal=model.predict_proba(X['calibration'])[:,1]
    threshold=select_detector_threshold(y['calibration'],pcal);thresholds[name]=threshold
    probs[name]={'calibration':pcal}
    detector_scores[name]={}
    for split in ('calibration','development','cal_missing','dev_missing'):
        prob=pcal if split=='calibration' else model.predict_proba(X[split])[:,1]
        probs[name][split]=prob
        pred=prob>=threshold
        tn,fp,fn,tp=confusion_matrix(y[split],pred,labels=[0,1]).ravel()
        detector_scores[name][split]={
            'n':int(len(pred)),'threshold':threshold,
            'F1':float(f1_score(y[split],pred,zero_division=0)),
            'AP':float(average_precision_score(y[split],prob)),
            'FAR':float(fp/(tn+fp)) if tn+fp else None,
            'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
        }
    print(name,'detector threshold calibration=',threshold,
          'F1 clean development=',round(detector_scores[name]['development']['F1'],4))
reference=json.loads(V2FILE.read_text(encoding='utf-8'))
if reference['status']!='DEVELOPMENT_ONLY_CORRECTED_V2_FILE2_UNTOUCHED':
    raise ValueError('JSON acuan bukan hasil v2 yang diharapkan; STOP.')
if reference['lock_manifest_sha256']!=lock['manifest']['sha256'] or reference['source_file1_sha256']!=lock['sources']['file1']['sha256']:
    raise ValueError('Protokol sumber/manifest berbeda dari v2; STOP.')
for name in trained:
    if abs(thresholds[name]-reference['detector_thresholds_from_calibration'][name])>1e-12:
        raise ValueError(f'Ambang detektor {name} berubah; STOP.')
    for split in ('calibration','development','cal_missing','dev_missing'):
        current=detector_scores[name][split]; earlier=reference['detector_results'][name][split]
        for key in ('n','TN','FP','FN','TP'):
            if current[key]!=earlier[key]:raise ValueError(f'{name} {split}: {key} berbeda dari v2; STOP.')
        for key in ('F1','AP','FAR'):
            if abs(current[key]-earlier[key])>1e-10:raise ValueError(f'{name} {split}: {key} berbeda dari v2; STOP.')
print('Rekonstruksi hasil detektor v2 lolos; file2 belum dibaca.')


## 5  Hitung stabilitas SHAP pada dua gangguan baru

Analisis dilakukan pada semua alert aktual, bukan hanya alert yang lolos gerbang. Rekaman sumber dan nilai r_x lama berasal dari CSV M2. Perbandingan dua gerbang adalah pemeriksaan diagnostik untuk ambang v2, sedangkan Jaccard dihitung dari SHAP baru yang terpisah.


In [ ]:
m2summary=json.loads(M2SUM.read_text(encoding='utf-8'))
if sha256(M2CSV)!=m2summary['per_instance_csv_sha256'] or sha256(V2FILE)!=m2summary['v2_reference_sha256']:
    raise ValueError('SHA sumber M2 atau v2 tidak cocok; STOP.')
if m2summary['lock_manifest_sha256']!=lock['manifest']['sha256'] or m2summary['source_file1_sha256']!=lock['sources']['file1']['sha256']:
    raise ValueError('Lock hasil M2 berbeda dari file1; STOP.')
m2=pd.read_csv(M2CSV,compression='gzip')
if len(m2)!=m2summary['per_instance_rows'] or m2.duplicated(['model','split','condition','source_row_zero_based']).any():
    raise ValueError('Jumlah atau indeks M2 per alert tidak konsisten; STOP.')

records=[];diagnostics={};NUM_REPEATS=2;SIGMA=.01
for name,model in trained.items():
    diagnostics[name]={};threshold=thresholds[name]
    frozen=reference['gate_thresholds_from_calibration'][name]
    explainer=shap.TreeExplainer(model)
    for split,condition,key in [('calibration','clean','calibration'),
                                ('calibration','synthetic_missing','cal_missing'),
                                ('development','clean','development'),
                                ('development','synthetic_missing','dev_missing')]:
        groupkey=split+'/'+condition
        candidate=np.flatnonzero(probs[name][key]>=threshold)
        expected=m2summary['gate2_reproduction_audit'][name][groupkey]
        if len(candidate)!=expected['candidate_alerts'] or len(candidate)>SHAP_MAX:
            raise ValueError('Jumlah kandidat berbeda atau melebihi batas; STOP.')
        frame=m2[(m2.model==name)&(m2.split==split)&(m2.condition==condition)].copy()
        frame=frame.set_index('source_row_zero_based').loc[parts[split][candidate]].reset_index()
        xmat=X[key][candidate]
        ytrue=y[key][candidate]
        prob=probs[name][key][candidate]
        if not np.array_equal(frame.y_true.to_numpy(int),ytrue):raise ValueError('Label kandidat tidak cocok; STOP.')
        if not np.allclose(frame.detector_probability.to_numpy(float),prob,atol=1e-10,rtol=0):
            raise ValueError('Prediksi kandidat tidak cocok; STOP.')
        md=margin_d(prob,threshold)
        if not np.allclose(frame.m_d_margin.to_numpy(float),md,atol=1e-10,rtol=0):
            raise ValueError('Margin kandidat tidak cocok; STOP.')
        rx=frame.r_x_local_shap.to_numpy(float)
        actions=gate2(md,rx,frozen['tau_d'],frozen['tau_x'])
        if not np.array_equal(frame.gate2_action.to_numpy(),actions):raise ValueError('Gerbang M2 tidak cocok; STOP.')
        base=tree_shap_positive(explainer,xmat)
        jac=[];retained=[]
        for rep in range(NUM_REPEATS):
            seed=stable_seed(name,split,condition,'heldout_shap',rep)
            xprobe=jitter_numeric_without_clipping(xmat,seed,SIGMA)
            jac.append(topk_jaccard(base,tree_shap_positive(explainer,xprobe),TOPK))
            retained.append(model.predict_proba(xprobe)[:,1]>=threshold)
        jac=np.column_stack(jac);retained=np.column_stack(retained)
        all_stable=retained.all(axis=1)
        indep=jac.mean(axis=1)
        margin_accept=md>=frozen['tau_d']
        two_accept=actions=='accept'
        frame['new_probe_1_topk_jaccard']=jac[:,0]
        frame['new_probe_2_topk_jaccard']=jac[:,1]
        frame['new_probe_1_alert_retained']=retained[:,0].astype(int)
        frame['new_probe_2_alert_retained']=retained[:,1].astype(int)
        frame['both_new_probes_alert_retained']=all_stable.astype(int)
        frame['mean_new_topk_jaccard']=indep
        frame['margin_only_accept']=margin_accept.astype(int)
        records.append(frame)
        unique_fail=margin_accept & ~two_accept
        # Rank correlation is computed only where both independent probes retain the alert.
        stable=frame[frame.both_new_probes_alert_retained==1]
        corr=stable[['r_x_local_shap','mean_new_topk_jaccard']].corr(method='spearman').iloc[0,1] if len(stable)>2 else np.nan
        diagnostics[name][groupkey]={
            'candidate_alerts':int(len(frame)),
            'both_independent_probes_retain_alert':int(all_stable.sum()),
            'mean_jaccard_given_stable_prediction':float(indep[all_stable].mean()) if all_stable.any() else None,
            'r_x_vs_independent_jaccard_spearman_given_stable_prediction':float(corr) if np.isfinite(corr) else None,
            'margin_only_accept':int(margin_accept.sum()),
            'margin_only_TP':int((margin_accept&(ytrue==1)).sum()),
            'margin_only_FP':int((margin_accept&(ytrue==0)).sum()),
            'two_domain_accept':int(two_accept.sum()),
            'two_domain_TP':int((two_accept&(ytrue==1)).sum()),
            'two_domain_FP':int((two_accept&(ytrue==0)).sum()),
            'SHAP_only_exclusions_after_margin':int(unique_fail.sum()),
            'SHAP_only_exclusions_that_are_true_alerts':int((unique_fail&(ytrue==1)).sum()),
            'stable_margin_accept_count':int((margin_accept&all_stable).sum()),
            'stable_two_domain_accept_count':int((two_accept&all_stable).sum()),
            'stable_margin_accept_mean_jaccard':float(indep[margin_accept&all_stable].mean()) if (margin_accept&all_stable).any() else None,
            'stable_two_domain_accept_mean_jaccard':float(indep[two_accept&all_stable].mean()) if (two_accept&all_stable).any() else None,
        }
        print(name,groupkey,'alert',len(frame),'SHAP-specific exclusions',int(unique_fail.sum()),
              'independent Jaccard on stable alert',diagnostics[name][groupkey]['mean_jaccard_given_stable_prediction'])

instances=pd.concat(records,ignore_index=True)
if len(instances)!=len(m2) or instances.duplicated(['model','split','condition','source_row_zero_based']).any():
    raise ValueError('Jumlah hasil M3 tidak lengkap; STOP.')


## 6  Arsip dan keputusan klaim

Apabila perbandingan gerbang masih tidak menunjukkan perbaikan yang dapat diatribusikan kepada M3, jangan mengklaim M3 meningkatkan ketepatan triage. Skor independen ini hanya dua gangguan dari keluarga yang sama: korelasi atau rerata Jaccard yang tinggi **belum** membuktikan fidelity penjelasan, risiko organisasi, manfaat manusia, atau validitas seluruh framework. Rancangan M3 dapat tetap menghasilkan bukti keterbatasan, alasan `warn`, dan tampilan penjelasan, dengan status eksperimental yang jelas. Validasi final di file2 dilakukan hanya sesudah desain dan aturan dikunci.


In [ ]:
if OUTFILE.exists() or EVIDFILE.exists():raise FileExistsError('Keluaran sudah ada; STOP.')
OUT.mkdir(parents=True,exist_ok=True)
instances.to_csv(EVIDFILE,index=False,compression={'method':'gzip','mtime':0})
result={
 'status':'M3_INDEPENDENT_EXPLANATION_DIAGNOSTIC_FILE1_ONLY',
 'source_file1_sha256':sha256(P1),
 'lock_manifest_sha256':sha256(MANIFEST),
 'v2_reference_sha256':sha256(V2FILE),
 'm2_summary_sha256':sha256(M2SUM),
 'm2_instances_sha256':sha256(M2CSV),
 'per_instance_csv_sha256':sha256(EVIDFILE),
 'per_instance_rows':len(instances),
 'config':{'seed':SEED,'top_k':TOPK,'noise_sigma_numeric_transformed':SIGMA,
           'independent_probe_count':NUM_REPEATS,'probe_clipping':False,
           'detector_retrained_to_reproduce_v2':True,
           'gate_threshold_updated':False,'file2_accessed':False},
 'diagnostics':diagnostics,
 'limitations':['Numerical perturbations of model inputs only; no external explanation ground truth',
                'Two independent probes belong to the same family as v2 original numeric-noise probe',
                'Explanation stability on perturbed but prediction-stable alerts is not fidelity or human usefulness',
                'v2 original explanation probe clips numeric input while new probes do not',
                'Development already inspected in prior v2 and M2 analysis: all comparisons exploratory',
                'No M4 threshold recalibration, no file2, M5 or M6 tested']}
OUTFILE.write_text(json.dumps(result,indent=2,ensure_ascii=False),encoding='utf-8')
print('Simpan hanya dua file ini:',OUTFILE,EVIDFILE)
print('Periksa juga kesimpulan yang tidak boleh diklaim pada bagian terakhir JSON.')
